# 04: Peer Comparison Analysis

This notebook compares a target mutual fund with its top 5 category peers identifying them automatically and ranking them by risk/return metrics.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from mf_analyser.analysis.comparison import discover_peers, compare_returns
from mf_analyser.config import DEFAULT_FUNDS, FUND_CODE_TO_NAME

# 1. Select a fund to compare (e.g., Parag Parikh Flexi Cap)
scheme_code = '122639'  
fund_label = FUND_CODE_TO_NAME.get(scheme_code, f"Scheme {scheme_code}")

print(f"Analysing peers for: {fund_label}")

# 2. Discover Peers
peers = discover_peers(scheme_code, limit=5)
peer_codes = [scheme_code] + [p[0] for p in peers]
peer_names = {scheme_code: fund_label}
for p_code, p_name in peers:
    peer_names[p_code] = p_name

print(f"Found {len(peers)} peers.")

In [ ]:
# 3. Compute Comparison Metrics
df = compare_returns(peer_codes)
df['fund_name'] = df['scheme_code'].map(peer_names)

# Display as table
df_display = df[['fund_name', '1Y_cagr_pct', '3Y_cagr_pct', '5Y_cagr_pct', '10Y_cagr_pct', 'max_drawdown_pct']]
df_display.sort_values("3Y_cagr_pct", ascending=False).style.format({
    "1Y_cagr_pct": "{:.2f}%",
    "3Y_cagr_pct": "{:.2f}%",
    "5Y_cagr_pct": "{:.2f}%",
    "10Y_cagr_pct": "{:.2f}%",
    "max_drawdown_pct": "{:.2f}%"
})

## Visualizing Performance (CAGR Comparison)

In [ ]:
# Melt for plotting
df_melted = df.melt(
    id_vars=['fund_name'], 
    value_vars=['1Y_cagr_pct', '3Y_cagr_pct', '5Y_cagr_pct', '10Y_cagr_pct'],
    var_name='Period',
    value_name='CAGR'
)
df_melted['Period'] = df_melted['Period'].str.replace('_cagr_pct', '')

fig = px.bar(
    df_melted, 
    x='Period', 
    y='CAGR', 
    color='fund_name', 
    barmode='group',
    title=f"CAGR Comparison: {fund_label} vs Peers",
    labels={'CAGR': 'Annualised Return (%)'},
    category_orders={'Period': ['1Y', '3Y', '5Y', '10Y']}
)
fig.show()

## Risk vs Return (Efficiency Frontier)

Comparing 3Y CAGR against Max Drawdown. Higher return with lower drawdown is better.

In [ ]:
fig = px.scatter(
    df, 
    x='max_drawdown_pct', 
    y='3Y_cagr_pct', 
    text='fund_name', 
    size='5Y_cagr_pct', # Bubble size by 5Y performance
    color='fund_name',
    title="Risk (Max Drawdown) vs Return (3Y CAGR)",
    labels={
        'max_drawdown_pct': 'Max Drawdown (%) - Risk',
        '3Y_cagr_pct': '3Y CAGR (%) - Return'
    }
)
fig.update_traces(textposition='top center')
fig.show()